# Module 1: Mutual Information Deep Dive

## Learning Objectives

By completing this notebook, you will:
1. Compute MI using three estimators (plug-in, KSG, sklearn adaptive) and understand when each applies
2. Compare MI rankings against Pearson correlation rankings on real data and identify where they diverge
3. Visualise nonlinear dependencies that MI detects but correlation misses
4. Benchmark MI estimation accuracy versus sample size

## Prerequisites
- Guide 01: Mutual Information for Feature Selection
- Basic NumPy, pandas, and matplotlib
- Familiarity with k-nearest neighbours

## Estimated Time: 15 minutes

---

## 1. Setup

All dependencies are from the standard scientific Python stack — no special installs needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import digamma
from scipy.stats import pearsonr, spearmanr
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Setup complete.")

## 2. Three MI Estimators Side by Side

We implement three estimators to develop intuition for how each works and where each fails.

### 2.1 Plug-In Estimator

Discretises both variables into bins, counts joint frequencies, and plugs them into the MI formula:

$$\hat{I}_{\text{plug-in}}(X;Y) = \sum_{i,j} \hat{p}(x_i, y_j) \log \frac{\hat{p}(x_i, y_j)}{\hat{p}(x_i)\hat{p}(y_j)}$$

The key weakness: bin width is arbitrary and bias grows with the number of bins.

In [ ]:
def mi_plugin(x: np.ndarray, y: np.ndarray, n_bins: int = 20) -> float:
    """
    Plug-in MI estimator via histogram discretisation with Miller-Madow correction.

    Parameters
    ----------
    x, y : 1-D arrays of floats
    n_bins : number of bins per dimension

    Returns
    -------
    float : estimated MI in nats
    """
    n = len(x)

    # Build joint histogram — quantile bins to handle skewed data
    x_edges = np.quantile(x, np.linspace(0, 1, n_bins + 1))
    y_edges = np.quantile(y, np.linspace(0, 1, n_bins + 1))

    # Ensure unique edges (duplicate quantiles collapse bins)
    x_edges = np.unique(x_edges)
    y_edges = np.unique(y_edges)

    joint_counts, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    joint = joint_counts / n

    # Marginal distributions
    px = joint.sum(axis=1, keepdims=True)   # shape (n_x_bins, 1)
    py = joint.sum(axis=0, keepdims=True)   # shape (1, n_y_bins)

    # MI: only sum over non-zero cells (log(0) = 0 by convention)
    mask = (joint > 0) & (px > 0) & (py > 0)
    mi = np.sum(joint[mask] * np.log(joint[mask] / (px * py)[mask]))

    # Miller-Madow bias correction: subtract (B - 1) / (2N)
    # where B is the number of occupied cells
    B = mask.sum()
    mi -= (B - 1) / (2 * n)

    return max(0.0, mi)


# Quick sanity check on perfectly correlated data
x_test = np.linspace(0, 1, 500)
y_perfect = x_test.copy()       # Y = X: maximum dependence
y_indep   = np.random.rand(500) # Y independent of X: MI should be near 0

print(f"Plugin MI (perfect dependence): {mi_plugin(x_test, y_perfect):.4f} nats")
print(f"Plugin MI (independence):        {mi_plugin(x_test, y_indep):.4f} nats")
print(f"(Independence baseline should be near 0; perfect dependence > 0)")

### 2.2 KSG Estimator

The KSG estimator uses $k$-nearest neighbours in the joint space to avoid binning:

$$\hat{I}_{\text{KSG}}(X;Y) = \psi(k) + \psi(N) - \left\langle \psi(n_x+1) + \psi(n_y+1) \right\rangle$$

It is asymptotically unbiased and the standard choice for continuous-continuous MI.

In [ ]:
def mi_ksg(x: np.ndarray, y: np.ndarray, k: int = 5) -> float:
    """
    KSG mutual information estimator (Algorithm 1, Kraskov et al. 2004).

    Parameters
    ----------
    x, y : 1-D arrays
    k     : number of nearest neighbours (default 5)

    Returns
    -------
    float : MI estimate in nats, clamped to >= 0
    """
    n = len(x)
    # Reshape for NearestNeighbors
    x2d = x.reshape(-1, 1)
    y2d = y.reshape(-1, 1)
    xy  = np.hstack([x2d, y2d])

    # Step 1: k-th neighbour distance in joint space (Chebyshev = max-norm)
    # +1 because NearestNeighbors includes the point itself
    knn_joint = NearestNeighbors(n_neighbors=k + 1, metric='chebyshev').fit(xy)
    dists_joint, _ = knn_joint.kneighbors(xy)
    eps = dists_joint[:, k]   # epsilon_i = distance to k-th neighbour

    # Step 2: count marginal neighbours within eps radius
    def count_marginal_neighbors(z2d, radii):
        knn_marg = NearestNeighbors(metric='chebyshev').fit(z2d)
        counts = np.array([
            len(knn_marg.radius_neighbors([z2d[i]], radius=radii[i],
                                          return_distance=False)[0]) - 1
            for i in range(len(z2d))
        ])
        return np.maximum(counts, 0)

    nx = count_marginal_neighbors(x2d, eps)
    ny = count_marginal_neighbors(y2d, eps)

    # Step 3: KSG formula
    mi = digamma(k) + digamma(n) - np.mean(digamma(nx + 1) + digamma(ny + 1))
    return max(0.0, float(mi))


print(f"KSG MI (perfect dependence): {mi_ksg(x_test, y_perfect):.4f} nats")
print(f"KSG MI (independence):        {mi_ksg(x_test, y_indep):.4f} nats")

### 2.3 sklearn Adaptive Estimator

scikit-learn's `mutual_info_regression` and `mutual_info_classif` use a KSG-based adaptive partitioning method — the production-ready version. We use it as the reference.

In [ ]:
def mi_sklearn(x: np.ndarray, y: np.ndarray, task: str = 'regression',
               k: int = 5) -> float:
    """
    sklearn MI estimator. Wraps mutual_info_classif or mutual_info_regression.

    Parameters
    ----------
    x : 1-D array (feature)
    y : 1-D array (target)
    task : 'regression' for continuous y, 'classification' for discrete y
    k    : number of neighbours (n_neighbors parameter)

    Returns
    -------
    float : MI estimate in nats
    """
    x2d = x.reshape(-1, 1)
    if task == 'regression':
        score = mutual_info_regression(x2d, y, n_neighbors=k, random_state=42)[0]
    else:
        score = mutual_info_classif(x2d, y, n_neighbors=k, random_state=42)[0]
    return float(score)


print(f"sklearn MI (perfect dependence): {mi_sklearn(x_test, y_perfect):.4f} nats")
print(f"sklearn MI (independence):        {mi_sklearn(x_test, y_indep):.4f} nats")

## 3. Where MI and Correlation Diverge

We generate four dependency patterns and show how Pearson correlation and MI respond differently.

The four patterns:
- **Linear:** $Y = X + \epsilon$ — both should detect it
- **Quadratic:** $Y = X^2 + \epsilon$ — Pearson blind; MI detects
- **Sinusoidal:** $Y = \sin(4\pi X) + \epsilon$ — Pearson near zero; MI detects
- **Independent:** $Y \perp X$ — both should be near zero

In [ ]:
n_samples = 800
X_base = np.random.uniform(-1, 1, n_samples)
noise_scale = 0.2
rng = np.random.default_rng(42)

# Four dependency types
patterns = {
    'Linear\n$Y = X + \\epsilon$':       X_base + rng.normal(0, noise_scale, n_samples),
    'Quadratic\n$Y = X^2 + \\epsilon$':   X_base**2 + rng.normal(0, noise_scale, n_samples),
    'Sinusoidal\n$Y = \\sin(4\\pi X) + \\epsilon$': np.sin(4 * np.pi * X_base) + rng.normal(0, noise_scale, n_samples),
    'Independent\n$Y \\perp X$':          rng.uniform(-1, 1, n_samples),
}

# Compute measures for each pattern
results = []
for name, Y in patterns.items():
    pearson_r, _ = pearsonr(X_base, Y)
    spearman_r, _ = spearmanr(X_base, Y)
    mi_val = mi_ksg(X_base, Y, k=5)
    results.append({
        'Pattern': name,
        'Pearson |r|': abs(pearson_r),
        'Spearman |r|': abs(spearman_r),
        'MI (KSG, nats)': mi_val,
    })

results_df = pd.DataFrame(results).set_index('Pattern')
print("Dependence measures across relationship types:")
print(results_df.round(4).to_string())

In [ ]:
# Visualise: scatter plots coloured by relationship type + bar chart of measures
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
pattern_list = list(patterns.items())
colours = sns.color_palette('husl', 4)

for col_idx, ((name, Y), row) in enumerate(zip(pattern_list, results)):
    ax_scatter = axes[0, col_idx]
    ax_scatter.scatter(X_base, Y, alpha=0.3, s=8, color=colours[col_idx])
    ax_scatter.set_title(name, fontsize=9)
    ax_scatter.set_xlabel('X')
    ax_scatter.set_ylabel('Y')

    ax_bar = axes[1, col_idx]
    vals  = [row['Pearson |r|'], row['Spearman |r|'], row['MI (KSG, nats)']]
    labels = ['Pearson\n|r|', 'Spearman\n|ρ|', 'MI\n(nats)']
    bar_colours = ['#e74c3c', '#3498db', '#2ecc71']
    ax_bar.bar(labels, vals, color=bar_colours, edgecolor='white', linewidth=1.2)
    ax_bar.set_ylim(0, max(1.0, max(vals) * 1.15))
    ax_bar.set_title('Measure values', fontsize=9)

plt.suptitle('MI vs Correlation: Where They Agree and Where They Diverge',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('mi_vs_correlation_patterns.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nKey finding: Pearson and Spearman are near zero for Quadratic and Sinusoidal")
print("patterns, while MI correctly identifies strong dependence.")

## 4. Real Data: MI vs Correlation Feature Rankings

We apply both MI and Pearson correlation to the diabetes dataset and find features where the rankings disagree. These disagreements point to nonlinear relationships — potentially important features that correlation-based filters would discard.

In [ ]:
# Load diabetes dataset (continuous target = disease progression measure)
diabetes = load_diabetes()
X_diab = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_diab = diabetes.target

# Standardise (important for KSG distance calculations)
scaler = StandardScaler()
X_diab_sc = pd.DataFrame(scaler.fit_transform(X_diab), columns=X_diab.columns)

print(f"Diabetes dataset: {X_diab.shape[0]} samples, {X_diab.shape[1]} features")
print(f"Target range: [{y_diab.min():.0f}, {y_diab.max():.0f}]")
print(f"\nFeatures: {list(X_diab.columns)}")

In [ ]:
# Compute MI scores (sklearn, uses adaptive KSG) and Pearson |r|
mi_scores = mutual_info_regression(X_diab_sc, y_diab, n_neighbors=5, random_state=42)
mi_series = pd.Series(mi_scores, index=X_diab.columns)

pearson_scores = X_diab_sc.corrwith(pd.Series(y_diab, index=X_diab_sc.index)).abs()

# Rank features (1 = most relevant)
mi_ranks      = mi_series.rank(ascending=False).astype(int)
pearson_ranks = pearson_scores.rank(ascending=False).astype(int)
rank_diff     = (mi_ranks - pearson_ranks).abs()

ranking_df = pd.DataFrame({
    'MI score': mi_series.round(4),
    'MI rank':  mi_ranks,
    'Pearson |r|': pearson_scores.round(4),
    'Pearson rank': pearson_ranks,
    'Rank disagreement': rank_diff,
}).sort_values('MI rank')

print("Feature rankings: MI vs Pearson correlation")
print(ranking_df.to_string())
print(f"\nLargest disagreement: '{rank_diff.idxmax()}' "
      f"(MI rank {mi_ranks[rank_diff.idxmax()]}, "
      f"Pearson rank {pearson_ranks[rank_diff.idxmax()]})")

In [ ]:
# Visualise ranking divergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: side-by-side bar chart sorted by MI rank
x_pos = np.arange(len(X_diab.columns))
features_by_mi = ranking_df.index.tolist()

axes[0].barh(x_pos + 0.2, ranking_df['MI score'],      0.4, label='MI (nats)',    color='#2ecc71')
axes[0].barh(x_pos - 0.2, ranking_df['Pearson |r|'],   0.4, label='Pearson |r|', color='#e74c3c')
axes[0].set_yticks(x_pos)
axes[0].set_yticklabels(features_by_mi)
axes[0].set_xlabel('Score (normalised to comparable range for display)')
axes[0].set_title('MI vs Pearson: Feature Scores')
axes[0].legend()

# Subplot 2: rank comparison scatter
for feat in X_diab.columns:
    colour = '#e74c3c' if rank_diff[feat] >= 3 else '#95a5a6'
    axes[1].scatter(pearson_ranks[feat], mi_ranks[feat], s=80, color=colour,
                    zorder=5)
    if rank_diff[feat] >= 3:
        axes[1].annotate(feat, (pearson_ranks[feat], mi_ranks[feat]),
                         textcoords='offset points', xytext=(5, 5), fontsize=8)

n_feats = len(X_diab.columns)
axes[1].plot([1, n_feats], [1, n_feats], 'k--', alpha=0.4, label='Perfect agreement')
axes[1].set_xlabel('Pearson rank (1 = most correlated)')
axes[1].set_ylabel('MI rank (1 = most informative)')
axes[1].set_title('Rank Agreement: MI vs Pearson\n(red = disagreement >= 3 positions)')
axes[1].legend()
axes[1].invert_yaxis()  # rank 1 at top
axes[1].invert_xaxis()

plt.tight_layout()
plt.savefig('ranking_divergence_diabetes.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Visualising Nonlinear Dependencies

For the feature(s) with the largest rank disagreement, we plot the actual relationship to understand *why* MI and correlation disagree.

In [ ]:
# Identify the two most disagreeing features
top_disagreements = rank_diff.nlargest(2).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, feat in zip(axes, top_disagreements):
    x_vals = X_diab_sc[feat].values
    y_vals = y_diab

    # Scatter
    ax.scatter(x_vals, y_vals, alpha=0.4, s=15, color='#3498db')

    # Nonparametric smoother (LOWESS via numpy poly fit per quantile)
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    y_sorted = y_vals[sort_idx]
    window = max(10, len(x_sorted) // 15)
    y_smooth = np.convolve(y_sorted, np.ones(window)/window, mode='valid')
    x_smooth = x_sorted[window//2: window//2 + len(y_smooth)]
    ax.plot(x_smooth, y_smooth, 'r-', linewidth=2.5, label='Trend (moving avg)')

    mi_val = mi_series[feat]
    pr_val = pearson_scores[feat]
    ax.set_title(f"Feature: '{feat}'\nMI = {mi_val:.4f} nats (rank {mi_ranks[feat]}) | "
                 f"Pearson |r| = {pr_val:.4f} (rank {pearson_ranks[feat]})",
                 fontsize=10)
    ax.set_xlabel(f'{feat} (standardised)')
    ax.set_ylabel('Target (disease progression)')
    ax.legend()

plt.suptitle('Features Where MI and Pearson Rankings Disagree Most',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('nonlinear_dependencies.png', dpi=120, bbox_inches='tight')
plt.show()
print("The nonlinear trend visible in these plots explains why MI assigns "
      "a different relevance score than Pearson correlation.")

## 6. Benchmark: MI Estimation Accuracy vs Sample Size

All MI estimators converge to the true value as $N \to \infty$. But at finite $N$, which estimator is most accurate and how much data do you need?

We use a synthetic bivariate Gaussian where the true MI is known analytically:

$$I(X;Y) = -\frac{1}{2} \log(1 - \rho^2)$$

for $(X, Y) \sim \mathcal{N}(0, \Sigma)$ with correlation $\rho$.

In [ ]:
def true_mi_gaussian(rho: float) -> float:
    """
    Analytical MI for bivariate standard Gaussian with correlation rho.
    I(X;Y) = -0.5 * log(1 - rho^2)  [nats]
    """
    return -0.5 * np.log(1 - rho**2)


rho = 0.7   # true correlation (moderate dependence)
true_mi = true_mi_gaussian(rho)
print(f"True MI for rho={rho}: {true_mi:.4f} nats")

# Benchmark across sample sizes
sample_sizes = [50, 100, 200, 500, 1000, 2000, 5000]
n_trials = 30  # repeat each to get mean ± std

results_benchmark = {name: [] for name in ['Plugin (20 bins)', 'KSG (k=5)', 'sklearn']}

cov = np.array([[1, rho], [rho, 1]])

for n_samp in sample_sizes:
    trial_results = {'Plugin (20 bins)': [], 'KSG (k=5)': [], 'sklearn': []}
    for trial in range(n_trials):
        xy = np.random.multivariate_normal([0, 0], cov, size=n_samp)
        x_t, y_t = xy[:, 0], xy[:, 1]

        trial_results['Plugin (20 bins)'].append(mi_plugin(x_t, y_t, n_bins=20))
        trial_results['KSG (k=5)'].append(mi_ksg(x_t, y_t, k=5))
        trial_results['sklearn'].append(mi_sklearn(x_t, y_t, task='regression', k=5))

    for name in results_benchmark:
        results_benchmark[name].append((
            np.mean(trial_results[name]),
            np.std(trial_results[name])
        ))

print("Benchmark complete.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colours_bench = {'Plugin (20 bins)': '#e74c3c', 'KSG (k=5)': '#3498db', 'sklearn': '#2ecc71'}

for est_name, values in results_benchmark.items():
    means = np.array([v[0] for v in values])
    stds  = np.array([v[1] for v in values])
    colour = colours_bench[est_name]

    axes[0].plot(sample_sizes, means, 'o-', label=est_name, color=colour, linewidth=2)
    axes[0].fill_between(sample_sizes, means - stds, means + stds,
                          alpha=0.15, color=colour)

axes[0].axhline(true_mi, color='black', linestyle='--', linewidth=1.5, label=f'True MI = {true_mi:.3f}')
axes[0].set_xscale('log')
axes[0].set_xlabel('Sample size (log scale)')
axes[0].set_ylabel('Estimated MI (nats)')
axes[0].set_title('MI Estimation Convergence (rho=0.7)')
axes[0].legend()

# Absolute bias plot
for est_name, values in results_benchmark.items():
    means = np.array([v[0] for v in values])
    bias = np.abs(means - true_mi)
    axes[1].plot(sample_sizes, bias, 'o-', label=est_name,
                  color=colours_bench[est_name], linewidth=2)

axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('Sample size (log scale)')
axes[1].set_ylabel('|Bias| (log scale)')
axes[1].set_title('Absolute Bias vs Sample Size')
axes[1].legend()

plt.suptitle('Which Estimator Converges Fastest?', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('mi_estimator_benchmark.png', dpi=120, bbox_inches='tight')
plt.show()

# Summary table at largest sample size
print("\nAt N=5000 (largest sample):")
for name, vals in results_benchmark.items():
    m, s = vals[-1]
    print(f"  {name}: {m:.4f} ± {s:.4f} (bias = {abs(m - true_mi):.4f})")
print(f"  True MI: {true_mi:.4f}")

## 7. Exercise: Bootstrap Confidence Intervals

**Task:** Implement a bootstrap confidence interval for MI. For each feature in the diabetes dataset, compute a 95% bootstrap CI using 200 resamples. Identify features whose CI lower bound is above zero — these are the features with statistically reliable MI estimates.

**Hints:**
- Resample with replacement: `idx = rng.choice(n, n, replace=True)`
- Compute MI on the resampled data
- The 95% CI uses the 2.5th and 97.5th percentiles of the bootstrap distribution

**Expected output:** A DataFrame with columns `MI`, `CI_lower`, `CI_upper` for each feature.

In [ ]:
# YOUR CODE HERE
# ---------------
# Implement bootstrap_mi_ci(x, y, n_bootstrap=200, alpha=0.05)
# Then apply to all features in X_diab_sc

def bootstrap_mi_ci(x: np.ndarray, y: np.ndarray,
                    n_bootstrap: int = 200, alpha: float = 0.05,
                    k: int = 5) -> tuple:
    """
    Bootstrap confidence interval for MI(x, y).

    Returns
    -------
    (point_estimate, lower_bound, upper_bound)
    """
    # TODO: implement bootstrap resampling
    pass


# TODO: apply to all features and print/plot the results
# ci_results = {col: bootstrap_mi_ci(X_diab_sc[col].values, y_diab)
#               for col in X_diab_sc.columns}

In [ ]:
# Reference implementation (run after attempting the exercise)
def bootstrap_mi_ci_solution(x: np.ndarray, y: np.ndarray,
                              n_bootstrap: int = 200, alpha: float = 0.05,
                              k: int = 5) -> tuple:
    """
    Bootstrap CI for MI.
    Point estimate uses all data; CI is from bootstrap distribution.
    """
    n = len(x)
    point = mi_sklearn(x, y, k=k)

    rng = np.random.default_rng(42)
    boot_mi = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        boot_mi.append(mi_sklearn(x[idx], y[idx], k=k))

    lower = np.percentile(boot_mi, 100 * alpha / 2)
    upper = np.percentile(boot_mi, 100 * (1 - alpha / 2))
    return point, lower, upper


# Apply to all diabetes features
print("Computing bootstrap CIs for all features (this takes ~30 seconds)...")
ci_data = {}
for col in X_diab_sc.columns:
    pt, lo, hi = bootstrap_mi_ci_solution(X_diab_sc[col].values, y_diab,
                                          n_bootstrap=100)  # fewer for speed
    ci_data[col] = {'MI': pt, 'CI_lower': lo, 'CI_upper': hi}

ci_df = pd.DataFrame(ci_data).T.sort_values('MI', ascending=False)
print("\n95% Bootstrap CIs for MI (diabetes features):")
print(ci_df.round(4).to_string())

# Features with CI lower bound > 0 are reliably informative
reliable = ci_df[ci_df['CI_lower'] > 0].index.tolist()
print(f"\nFeatures with reliable MI (CI lower > 0): {reliable}")

## 8. Summary

### Key Takeaways

1. **MI detects all dependencies.** Pearson correlation is blind to nonlinear relationships (quadratic, sinusoidal). MI correctly identifies them.

2. **KSG is the standard estimator** for continuous features. sklearn's `mutual_info_regression` implements it with adaptive partitioning — use it in practice.

3. **Sample size matters.** KSG needs ~500 samples for reliable MI estimates of weak dependencies. For small datasets, bootstrap CIs reveal which MI estimates are trustworthy.

4. **Ranking divergence = nonlinear signal.** Features where MI and Pearson rankings disagree strongly are candidates for further investigation — the MI says they're informative, and the correlation says linear structure is weak, which together imply nonlinear predictive structure.

### What's Next

Notebook 02 — mRMR from scratch: adding redundancy awareness to MI-based feature selection.

### Additional Resources
- Kraskov, A. et al. (2004). *Estimating mutual information.* Physical Review E, 69(6).
- sklearn docs: [mutual_info_regression](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_regression.html)
- Ross, B.C. (2014). *Mutual information between discrete and continuous data sets.* PLOS ONE.